In [1]:
import clickhouse_connect

from clickhouse_connect.driver.tools import insert_file
import pyarrow
import pandas as pd


In [2]:
client = clickhouse_connect.get_client(host='localhost', username='default', password='')

In [3]:
df = pd.read_parquet('../silver/labeled_training_set_no_tensor.parquet')

In [4]:
list_of_dfs = [df]*10

In [5]:
union_all_df = pd.concat(list_of_dfs, ignore_index=True)

In [6]:
client.command("""
create table idnet10k
(
    name             text,
    imageData      text,
    prelabeled_fraud integer,
    id               text,
    person_name      text,
    address          text,
    birthday         text,
    gender           text,
    ethnicity        Nullable(text),
    class            text,
    issue_date       text,
    expire_date      text,
    eye_color        text,
    hair_color       Nullable(text),
    is_donor         text,
    is_veteran       text,
    license_number   text,
    height_inches    integer,
    weight_lbs       integer
)
    ENGINE = MergeTree
ORDER BY name
SETTINGS allow_nullable_key = 1;
""")

In [7]:
client.insert_df('idnet10k', union_all_df)

In [8]:
client.command("""select count(name) from idnet10k""")

9980